# Athena Audit Dashboard

Interactive notebook for querying Athena audit data (IDC workgroups).

**Run cells 1-4 in order**, then use the interactive form.

Requirements: `pip install pyathena pandas ipywidgets plotly cihi_auth`

In [ ]:
# === Step 1: IDC Authentication ===
from cihi_auth.jupyter_helper import authenticate, get_session

authenticate()
session = get_session(profile='default')
print('Authenticated via TIP')

In [ ]:
# === Step 2: Connect to Athena ===
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import plotly.express as px

from pyathena import connect
from datetime import date, timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output

REGION = 'us-east-1'
DATABASE = 'athena_events'
TABLE = 'events'
WORKGROUP = 'idc-wg'

creds = session.get_credentials().get_frozen_credentials()
conn = connect(
    region_name=REGION,
    work_group=WORKGROUP,
    schema_name=DATABASE,
    aws_access_key_id=creds.access_key,
    aws_secret_access_key=creds.secret_key,
    aws_session_token=creds.token,
)
print('Connected to Athena (' + WORKGROUP + ')')

In [ ]:
# === Step 3: Load filter options ===
users_df = pd.read_sql(
    'SELECT DISTINCT source_identity FROM ' + TABLE + ' WHERE source_identity IS NOT NULL ORDER BY 1', conn
)
wg_df = pd.read_sql(
    'SELECT DISTINCT workgroup FROM ' + TABLE + ' WHERE workgroup IS NOT NULL ORDER BY 1', conn
)
user_list = users_df['source_identity'].dropna().tolist()
wg_list = wg_df['workgroup'].dropna().tolist()
print('Loaded ' + str(len(user_list)) + ' users, ' + str(len(wg_list)) + ' workgroups')

---
## Interactive Audit Search
Use the form below to search audit data. The **User Queries** tab is the primary view.

In [ ]:
# === Step 4: Interactive Dashboard ===

# --- Widgets ---
style = {'description_width': '130px'}
dd_layout = widgets.Layout(width='300px')

w_from = widgets.DatePicker(description='From Date:', value=date.today() - timedelta(days=7), style=style)
w_to = widgets.DatePicker(description='To Date:', value=date.today(), style=style)
w_user = widgets.Dropdown(description='User:', options=['All'] + user_list, value='All', style=style, layout=dd_layout)
w_workgroup = widgets.Dropdown(description='Workgroup:', options=['All'] + wg_list, value='All', style=style, layout=dd_layout)
w_status = widgets.Dropdown(description='Status:', options=['All', 'SUCCEEDED', 'FAILED', 'CANCELLED'], value='All', style=style, layout=dd_layout)
w_query_text = widgets.Text(description='Query Contains:', placeholder='e.g. SELECT, clients', style=style, layout=widgets.Layout(width='400px'))
w_limit = widgets.IntSlider(description='Max Rows:', value=200, min=10, max=5000, step=10, style=style)

btn_user_queries = widgets.Button(description='User Queries', button_style='primary', icon='search',
                                  layout=widgets.Layout(width='180px', height='38px'))
btn_dashboard = widgets.Button(description='Dashboard', button_style='info', icon='bar-chart',
                               layout=widgets.Layout(width='180px', height='38px'))
btn_export = widgets.Button(description='Export CSV', button_style='success', icon='download',
                            layout=widgets.Layout(width='180px', height='38px'))

output = widgets.Output()
current_df = None

# --- Layout ---
form = widgets.VBox([
    widgets.HBox([w_from, w_to]),
    widgets.HBox([w_user, w_workgroup, w_status]),
    widgets.HBox([w_query_text, w_limit]),
    widgets.HBox([btn_user_queries, btn_dashboard, btn_export]),
])
display(form, output)

# --- Helper: build WHERE clause ---
def where_clause():
    conds = ["day BETWEEN '" + str(w_from.value) + "' AND '" + str(w_to.value) + "'"]
    if w_user.value != 'All':
        conds.append("source_identity = '" + w_user.value + "'")
    if w_workgroup.value != 'All':
        conds.append("workgroup = '" + w_workgroup.value + "'")
    if w_status.value != 'All':
        conds.append("status = '" + w_status.value + "'")
    if w_query_text.value.strip():
        conds.append("LOWER(query) LIKE '%" + w_query_text.value.strip().lower() + "%'")
    return ' AND '.join(conds)

# ──────────────────────────────────────────────
# USER QUERIES - the primary view
# ──────────────────────────────────────────────
def on_user_queries(btn):
    global current_df
    with output:
        clear_output(wait=True)
        print('Querying...')
        sql = (
            'SELECT event_time, source_identity, workgroup, status, '
            'SUBSTR(query, 1, 300) AS query_text, '
            'database, ROUND(data_scanned / 1048576.0, 2) AS data_mb '
            'FROM ' + TABLE + ' '
            'WHERE ' + where_clause() + ' '
            'ORDER BY event_time DESC '
            'LIMIT ' + str(w_limit.value)
        )
        try:
            current_df = pd.read_sql(sql, conn)
            if current_df.empty:
                print('No results. Try adjusting filters.')
                return
            n = len(current_df)
            print(str(n) + ' queries found')
            print()
            # Summary per status
            status_summary = current_df.groupby('status').agg(
                count=('status', 'size'),
                total_mb=('data_mb', 'sum')
            ).reset_index()
            print('--- Status Summary ---')
            for _, r in status_summary.iterrows():
                print('  ' + str(r['status']) + ': ' + str(r['count']) + ' queries, ' + str(round(r['total_mb'], 2)) + ' MB scanned')
            print()
            # Show table
            display(current_df.style.set_properties(**{
                'text-align': 'left', 'white-space': 'pre-wrap', 'max-width': '500px'
            }).set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}]))
        except Exception as e:
            print('Query failed: ' + str(e))

# ──────────────────────────────────────────────
# DASHBOARD - charts
# ──────────────────────────────────────────────
def on_dashboard(btn):
    with output:
        clear_output(wait=True)
        w = where_clause()
        print('Loading dashboard...')

        # Summary stats
        s = pd.read_sql(
            'SELECT COUNT(*) AS total, '
            'COUNT(DISTINCT source_identity) AS users, '
            'COUNT(DISTINCT workgroup) AS workgroups, '
            "SUM(CASE WHEN status='SUCCEEDED' THEN 1 ELSE 0 END) AS ok, "
            "SUM(CASE WHEN status='FAILED' THEN 1 ELSE 0 END) AS fail, "
            "SUM(CASE WHEN status='CANCELLED' THEN 1 ELSE 0 END) AS cancel, "
            'ROUND(SUM(data_scanned)/1073741824.0,3) AS gb '
            'FROM ' + TABLE + ' WHERE ' + w, conn
        ).iloc[0]
        print('=' * 55)
        print('  AUDIT SUMMARY: ' + str(w_from.value) + ' to ' + str(w_to.value))
        print('=' * 55)
        print('  Total Queries:   ' + str(int(s['total'])))
        print('  Unique Users:    ' + str(int(s['users'])))
        print('  Workgroups:      ' + str(int(s['workgroups'])))
        print('  Succeeded:       ' + str(int(s['ok'])))
        print('  Failed:          ' + str(int(s['fail'])))
        print('  Cancelled:       ' + str(int(s['cancel'])))
        print('  Data Scanned:    ' + str(s['gb']) + ' GB')
        print('=' * 55)
        print()

        # Queries per user
        udf = pd.read_sql(
            'SELECT COALESCE(source_identity, user_identity_type) AS user_name, '
            'status, COUNT(*) AS cnt '
            'FROM ' + TABLE + ' WHERE ' + w + ' '
            'GROUP BY 1, 2 ORDER BY cnt DESC LIMIT 50', conn
        )
        if not udf.empty:
            fig = px.bar(udf, x='user_name', y='cnt', color='status',
                         title='Queries Per User by Status',
                         labels={'user_name': 'User', 'cnt': 'Count', 'status': 'Status'},
                         color_discrete_map={'SUCCEEDED': '#2ecc71', 'FAILED': '#e74c3c', 'CANCELLED': '#f39c12'},
                         barmode='stack')
            fig.update_layout(xaxis_tickangle=-45, height=400)
            fig.show()

        # Status pie
        sdf = pd.read_sql(
            "SELECT COALESCE(status,'UNKNOWN') AS status, COUNT(*) AS cnt "
            'FROM ' + TABLE + ' WHERE ' + w + ' '
            "GROUP BY 1", conn
        )
        if not sdf.empty:
            fig = px.pie(sdf, names='status', values='cnt', title='Status Breakdown',
                         color='status',
                         color_discrete_map={'SUCCEEDED': '#2ecc71', 'FAILED': '#e74c3c', 'CANCELLED': '#f39c12', 'UNKNOWN': '#95a5a6'})
            fig.update_traces(textinfo='label+percent+value')
            fig.update_layout(height=400)
            fig.show()

        # Timeline
        tdf = pd.read_sql(
            'SELECT day, workgroup, COUNT(*) AS cnt '
            'FROM ' + TABLE + ' WHERE ' + w + ' '
            'GROUP BY 1, 2 ORDER BY 1', conn
        )
        if not tdf.empty:
            fig = px.bar(tdf, x='day', y='cnt', color='workgroup',
                         title='Queries Per Day', barmode='stack',
                         labels={'day': 'Day', 'cnt': 'Count', 'workgroup': 'Workgroup'})
            fig.update_layout(height=400)
            fig.show()

        # Data scanned
        ddf = pd.read_sql(
            'SELECT workgroup, COUNT(*) AS cnt, '
            'ROUND(SUM(data_scanned)/1073741824.0,3) AS gb '
            'FROM ' + TABLE + ' WHERE ' + w + ' AND workgroup IS NOT NULL '
            'GROUP BY 1 ORDER BY gb DESC', conn
        )
        if not ddf.empty:
            fig = px.bar(ddf, x='workgroup', y='gb', text='cnt',
                         title='Data Scanned Per Workgroup',
                         labels={'workgroup': 'Workgroup', 'gb': 'GB Scanned', 'cnt': 'Queries'},
                         color='gb', color_continuous_scale='Reds')
            fig.update_traces(texttemplate='%{text} queries', textposition='outside')
            fig.update_layout(height=400)
            fig.show()

# ──────────────────────────────────────────────
# EXPORT
# ──────────────────────────────────────────────
def on_export(btn):
    with output:
        if current_df is None or current_df.empty:
            print('No data to export. Run User Queries first.')
            return
        path = 'audit_export.csv'
        current_df.to_csv(path, index=False)
        print('Exported ' + str(len(current_df)) + ' rows to ' + path)

btn_user_queries.on_click(on_user_queries)
btn_dashboard.on_click(on_dashboard)
btn_export.on_click(on_export)
print('Ready. Select a user and click User Queries.')

---
## Ad-Hoc SQL
Modify and run the cell below to execute any custom query.

In [ ]:
# === Custom SQL ===
custom_sql = """
SELECT source_identity, workgroup, status, COUNT(*) AS cnt,
       ROUND(SUM(data_scanned) / 1048576.0, 2) AS total_mb
FROM events
WHERE day >= '2026-02-28'
  AND source_identity IS NOT NULL
GROUP BY source_identity, workgroup, status
ORDER BY cnt DESC
LIMIT 50
"""

result = pd.read_sql(custom_sql, conn)
display(result)